# Análisis de repuestos · consumo a buses (12, 18 y 9 m) y consumo interno TVL

Clasifica cada **llave** (grupo de SKU de `Bd_Llaves_Actualizado.xlsx`) por su consumo histórico y su abastecimiento:

1. **Consumo a buses** (kardex 91 salida − 92 devolución, TVL + TATSA), con un análisis completo por cada **tamaño de bus** (12, 18 y 9 m) y otro de **toda la flota** (los tres tamaños juntos).
2. **ABC‑XYZ:** ABC por valor consumido en los últimos 12 meses cerrados (A ≤ 80 %, B ≤ 95 %, C resto); XYZ por variabilidad mensual (CV < 0,5 = X, ≤ 1,0 = Y, > 1,0 = Z). Por cada tamaño y para toda la flota.
3. **Abastecimiento:** lead time (días del RQ a la primera recepción, mediana y P90) y su tramo, stock de seguridad y **punto de reorden**, stock hoy, tránsito, cobertura.
4. **Compras:** importación (OC de importación o proveedor extranjero), última compra (fecha, OC, proveedor, moneda, precio), precio promedio 12 meses, proveedor principal y número de proveedores.
5. Lo mismo para el **consumo interno de TVL** (kardex 10: insumos de limpieza, EPP, suministros).

| Concepto | Fórmula |
|---|---|
| Demanda mensual | promedio de los últimos 12 meses cerrados (unidades) |
| Stock de seguridad | Z × desviación mensual × √(lead time / 30). Z según ABC: A 1,65 (95 %), B 1,28 (90 %), C 1,04 (85 %) |
| Punto de reorden | demanda mensual × lead time / 30 + stock de seguridad |
| Posición | stock hoy + tránsito (OC pendiente + RQ sin OC) |
| Estado | BAJO PUNTO DE REORDEN si posición < punto de reorden (A y B, o C‑X / C‑Y) · C‑Z: COMPRA A PEDIDO (consumo bajo y esporádico: no conviene tener stock) · SOBRESTOCK si el stock cubre más de 6 meses · SIN ROTACIÓN si hay stock sin consumo en 12 meses |

Valores del consumo a costo del kardex (soles sin IGV). Precios de compra en soles con IGV. Solo lectura en Odoo.
**Orden de ejecución:** `Tablas_Actual_BI` (kardex y flota) → `Saldo_Disponible_Real` (tránsito) → este notebook.

##### Librerías

In [ ]:
import xmlrpc.client
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from openpyxl.styles import Font, PatternFill, Alignment, Border
from openpyxl.utils import get_column_letter

##### Parámetros

In [ ]:
EMPRESAS = {"TVL": 1, "TATSA": 2}
USUARIO = "JIM"
FLOTAS = ["12 METROS", "18 METROS", "9 METROS"]
HISTORIA_DESDE = "2024-01-01"
MESES_ANALISIS = 12                              # Ventana de ABC, XYZ y demanda (meses cerrados)
ABC_CORTES = (0.80, 0.95)                        # A <= 80 % del valor, B <= 95 %
XYZ_CORTES = (0.50, 1.00)                        # CV mensual: X < 0,5, Y <= 1,0, Z > 1,0
Z_SERVICIO = {"A": 1.65, "B": 1.28, "C": 1.04}   # Nivel de servicio 95 % / 90 % / 85 %
TRAMOS_LEAD_TIME = [(15, "0-15 días"), (30, "16-30 días"), (60, "31-60 días"), (90, "61-90 días"), (10**6, "Más de 90 días")]
MESES_SOBRESTOCK = 6
PRODUCTOS_EXCLUIDOS = {"G3370009"}
FAMILIAS_EXCLUIDAS = {"BUSES"}                   # Buses completos: no son repuesto

ARCHIVO_KARDEX = "Información_Proyecto_BI/Fact_Kardex.parquet"
ARCHIVO_FLOTA = "Información_Proyecto_BI/Dim_Flota.parquet"
ARCHIVO_LLAVES = "Bd_Llaves_Actualizado.xlsx"
ARCHIVO_SALDO = "Saldo_Disponible_Real_2024_2026.xlsx"
ARCHIVO_EXCEL = "Analisis_Repuestos.xlsx"
EXPORTAR_EXCEL = True

HOY = pd.Timestamp.today().normalize()
ULTIMO_CERRADO = HOY.to_period("M") - 1                      # último mes cerrado
VENTANA = pd.period_range(ULTIMO_CERRADO - MESES_ANALISIS + 1, ULTIMO_CERRADO, freq="M")
print(f"Ventana de análisis: {VENTANA[0]} a {VENTANA[-1]}")

##### Conexión

In [ ]:
load_dotenv()
url, db = os.getenv("URL_ODOO"), os.getenv("BD_ODOO")
username, password = os.getenv(f"USER_{USUARIO}"), os.getenv(f"PASSWORD_{USUARIO}")
uid = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/common").authenticate(db, username, password, {})
models = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/object", allow_none=True)
print(f"uid conectado : {uid}")


def odoo(modelo, metodo, argumentos, opciones=None, empresas=None):
    opciones = dict(opciones or {})
    contexto = {"lang": "es_PE", "tz": "America/Lima", **opciones.pop("context", {})}
    if empresas:
        contexto["allowed_company_ids"] = empresas
    opciones["context"] = contexto
    return models.execute_kw(db, uid, password, modelo, metodo, argumentos, opciones)


def leer_todo(modelo, dominio, campos, empresas, lote=5000, contexto=None):
    filas, desplazamiento = [], 0
    while True:
        bloque = odoo(modelo, "search_read", [dominio],
                      {"fields": campos, "limit": lote, "offset": desplazamiento, "order": "id", "context": contexto or {}}, empresas)
        filas += bloque
        if len(bloque) < lote:
            return filas
        desplazamiento += lote


def leer_ids(modelo, ids, campos, empresas, lote=2000):
    ids, filas = list(ids), []
    for i in range(0, len(ids), lote):
        filas += odoo(modelo, "read", [ids[i:i + lote]], {"fields": campos}, empresas)
    return filas


def m2o_id(valor):
    return valor[0] if isinstance(valor, list) and valor else None


def m2o_nombre(valor):
    return valor[1] if isinstance(valor, list) and len(valor) == 2 else None

##### 1. Maestros: productos, llaves, unidades y flota (tamaño de bus)

In [ ]:
llaves = pd.read_excel(ARCHIVO_LLAVES, engine="calamine")
llaves = llaves[["Cod_Producto", "Cod_Grupo_Sku", "Producto_Grupo"]].dropna(subset=["Cod_Producto"]).drop_duplicates("Cod_Producto")
productos = leer_todo("product.product", [], ["default_code", "name", "detailed_type", "categ_id", "uom_id"], [1, 2], contexto={"active_test": False})
df_productos = pd.DataFrame([{
    "product_id": p["id"], "Cod_Producto": p["default_code"] or None, "Producto": p["name"], "Tipo_Producto": p["detailed_type"],
    "Familia": (m2o_nombre(p["categ_id"]) or "").split("/")[0].strip() or None, "Categoria": m2o_nombre(p["categ_id"]),
    "udm_producto": m2o_id(p["uom_id"]), "UdM": m2o_nombre(p["uom_id"]),
} for p in productos]).merge(llaves, on="Cod_Producto", how="left")
df_productos["Llave"] = df_productos["Cod_Grupo_Sku"].fillna(df_productos["Cod_Producto"])
df_productos["Producto_Grupo"] = df_productos["Producto_Grupo"].fillna(df_productos["Producto"])
PRODUCTO = df_productos.set_index("product_id").to_dict("index")
LLAVE_DE_CODIGO = df_productos.dropna(subset=["Cod_Producto"]).drop_duplicates("Cod_Producto").set_index("Cod_Producto")["Llave"].to_dict()
INFO_LLAVE = df_productos.dropna(subset=["Cod_Producto"]).sort_values("Cod_Producto").drop_duplicates("Llave").set_index("Llave")[
    ["Producto_Grupo", "Familia", "Categoria", "UdM", "Tipo_Producto"]]
CODIGOS_LLAVE = df_productos.dropna(subset=["Cod_Producto"]).groupby("Llave")["Cod_Producto"].agg(lambda s: ", ".join(sorted(set(s))[:8]))

UDM = {u["id"]: {"factor": u["factor"] or 1.0, "categoria": m2o_id(u["category_id"])}
       for u in odoo("uom.uom", "search_read", [[]], {"fields": ["factor", "category_id"], "context": {"active_test": False}})}


def a_udm_producto(cantidad, udm_origen, product_id):
    destino = PRODUCTO.get(product_id, {}).get("udm_producto")
    o, d = UDM.get(udm_origen), UDM.get(destino)
    if not o or not d or udm_origen == destino or o["categoria"] != d["categoria"]:
        return cantidad
    return cantidad / o["factor"] * d["factor"]


"""
    FLOTA: TAMAÑO DE BUS (Dim_Flota)
"""
df_flota = pd.read_parquet(ARCHIVO_FLOTA)
BUSES = df_flota[df_flota["Tamaño_Bus"].isin(FLOTAS)].groupby("Tamaño_Bus")["Cod_Bus"].nunique().to_dict()
BUSES["TODA LA FLOTA"] = sum(BUSES.values())
print("Buses por tamaño:", BUSES)


##### 2. Consumo del kardex

- **Buses:** 91 (salida a bus) − 92 (devolución del bus), de los almacenes de TVL y TATSA, solo buses de 12, 18 y 9 m. Valor = costo del kardex.
- **Consumo interno TVL:** 10 (salida a consumo interno) del almacén de TVL.
- Unidades y valor por llave + mes; los meses sin consumo cuentan como 0.

In [ ]:
kardex = pd.read_parquet(ARCHIVO_KARDEX)
kardex = kardex[(kardex["Fecha"] >= HISTORIA_DESDE) & ~kardex["Cod_Producto"].isin(PRODUCTOS_EXCLUIDOS)].copy()
kardex["Llave"] = kardex["Cod_Producto"].map(LLAVE_DE_CODIGO).fillna(kardex["Cod_Producto"])
kardex = kardex[~kardex["Llave"].map(INFO_LLAVE["Familia"]).isin(FAMILIAS_EXCLUIDAS)]
kardex["Periodo_Mes"] = kardex["Fecha"].dt.to_period("M")
kardex["Cantidad"] = kardex["Cant_Salida"] - kardex["Cant_Ingreso"]
kardex["Valor"] = kardex["Valor_Salida"] - kardex["Valor_Ingreso"]

buses = kardex[kardex["Cod_Movimiento"].isin([91, 92])].merge(df_flota, on="Cod_Bus", how="left")
buses = buses[buses["Tamaño_Bus"].isin(FLOTAS)].copy()
interno = kardex[(kardex["Cod_Movimiento"] == 10) & (kardex["Empresa"] == "Tvl")].copy()
MESES_HIST = pd.period_range(pd.Period(HISTORIA_DESDE[:7], "M"), ULTIMO_CERRADO, freq="M")
print(f"Buses: {len(buses):,} movimientos | {buses['Llave'].nunique():,} llaves | consumo interno TVL: {len(interno):,} movimientos | {interno['Llave'].nunique():,} llaves")

##### 3. Compras, lead time e importación

- **OC confirmadas** de TVL y TATSA desde 2024: cantidad en UdM del producto, valor en soles con IGV, proveedor, moneda.
- **Importación:** OC marcada como importación en Odoo o proveedor extranjero (comprar en dólares a un proveedor local no cuenta).
- **Lead time:** días del RQ (o de la OC si no tiene RQ) a la primera recepción de esa línea; mediana y percentil 90 por llave.

In [ ]:
def leer_compras(empresa_id, nombre):
    ordenes = {o["id"]: o for o in leer_todo("purchase.order",
               [["company_id", "=", empresa_id], ["state", "in", ["purchase", "done"]], ["date_order", ">=", HISTORIA_DESDE]],
               ["name", "currency_rate", "currency_id", "date_approve", "date_order", "requisition_id", "importacion_check", "partner_id"], [empresa_id])}
    paises = {p["id"]: m2o_nombre(p["country_id"]) for p in leer_ids("res.partner", {m2o_id(o["partner_id"]) for o in ordenes.values()}, ["country_id"], [empresa_id])}
    rqs = {r["id"]: r["create_date"][:10] for r in leer_ids("purchase.requisition", {m2o_id(o["requisition_id"]) for o in ordenes.values() if o["requisition_id"]},
                                                            ["create_date"], [empresa_id])}
    lineas = leer_todo("purchase.order.line", [["order_id", "in", list(ordenes)], ["product_id", "!=", False]],
                       ["order_id", "product_id", "product_uom", "product_qty", "price_total"], [empresa_id])
    filas = []
    for l in lineas:
        o, p = ordenes[m2o_id(l["order_id"])], m2o_id(l["product_id"])
        cant = a_udm_producto(l["product_qty"] or 0.0, m2o_id(l["product_uom"]), p)
        if cant <= 0 or not l["price_total"]:
            continue
        pais = paises.get(m2o_id(o["partner_id"]))
        valor = l["price_total"] / (o["currency_rate"] or 1.0)
        filas.append({"Empresa": nombre, "po_line_id": l["id"], "Cod_OC": o["name"], "Fecha_OC": (o["date_approve"] or o["date_order"])[:10],
                      "Fecha_RQ": rqs.get(m2o_id(o["requisition_id"])), "Llave": PRODUCTO.get(p, {}).get("Llave"),
                      "Cod_Producto": PRODUCTO.get(p, {}).get("Cod_Producto"), "Proveedor": m2o_nombre(o["partner_id"]), "Pais": pais,
                      "Moneda": m2o_nombre(o["currency_id"]), "Cantidad": cant, "Valor": valor, "Precio": valor / cant,
                      "Importacion": bool(o["importacion_check"]) or pais not in (None, "Perú", "Peru")})
    return pd.DataFrame(filas)


def primeras_recepciones(empresa_id):
    movs = leer_todo("stock.move", [["company_id", "=", empresa_id], ["state", "=", "done"], ["purchase_line_id", "!=", False],
                                    ["location_id.usage", "=", "supplier"], ["date", ">=", HISTORIA_DESDE]], ["purchase_line_id", "date"], [empresa_id])
    return pd.DataFrame([{"po_line_id": m2o_id(m["purchase_line_id"]), "Fecha": m["date"][:10]} for m in movs]).groupby("po_line_id")["Fecha"].min()


compras = pd.concat([leer_compras(e, n) for n, e in EMPRESAS.items()], ignore_index=True).sort_values(["Fecha_OC", "Cod_OC"])
recep = pd.concat([primeras_recepciones(e) for e in EMPRESAS.values()])
compras = compras.join(recep.rename("Fecha_Recepcion"), on="po_line_id")
inicio = pd.to_datetime(compras["Fecha_RQ"].fillna(compras["Fecha_OC"]))
compras["Lead_Time"] = (pd.to_datetime(compras["Fecha_Recepcion"]) - inicio).dt.days
compras.loc[compras["Lead_Time"] < 0, "Lead_Time"] = np.nan
LT_GENERAL = compras["Lead_Time"].median()

desde_12m = str(VENTANA[0].start_time.date())
c12 = compras[compras["Fecha_OC"] >= desde_12m]
c24 = compras[compras["Fecha_OC"] >= str((VENTANA[0] - 12).start_time.date())]
ultima = compras.groupby("Llave").last()
principal = c24.groupby(["Llave", "Proveedor"])["Valor"].sum().reset_index().sort_values("Valor", ascending=False)
part = principal.groupby("Llave")["Valor"].transform("sum")
principal["Participacion"] = principal["Valor"] / part
principal = principal.drop_duplicates("Llave").set_index("Llave")

COMPRAS_LLAVE = pd.DataFrame({
    "Importacion": compras.groupby("Llave")["Importacion"].any().map({True: "Sí", False: "No"}),
    "Lead_Time_Mediana": compras.groupby("Llave")["Lead_Time"].median(),
    "Lead_Time_P90": compras.groupby("Llave")["Lead_Time"].quantile(0.9),
    "Recepciones_Medidas": compras.groupby("Llave")["Lead_Time"].count(),
    "Fecha_Ultima_Compra": ultima["Fecha_OC"], "Ultima_OC": ultima["Cod_OC"], "Ultimo_Proveedor": ultima["Proveedor"],
    "Moneda_Ultima": ultima["Moneda"], "Precio_Ultimo": ultima["Precio"],
    "Precio_Prom_12M": c12.groupby("Llave")["Valor"].sum() / c12.groupby("Llave")["Cantidad"].sum(),
    "Compras_Und_12M": c12.groupby("Llave")["Cantidad"].sum(), "Compras_Valor_12M": c12.groupby("Llave")["Valor"].sum(),
    "OC_12M": c12.groupby("Llave")["Cod_OC"].nunique(),
    "Proveedor_Principal": principal["Proveedor"], "Participacion_Principal": principal["Participacion"],
    "N_Proveedores_24M": c24.groupby("Llave")["Proveedor"].nunique(),
    "Proveedores_24M": c24.groupby("Llave")["Proveedor"].agg(lambda s: " | ".join(s.value_counts().index[:4])),
})
print(f"Líneas de OC: {len(compras):,} | llaves con compras: {len(COMPRAS_LLAVE):,} | de importación: {(COMPRAS_LLAVE['Importacion'] == 'Sí').sum():,} | "
      f"lead time general (mediana): {LT_GENERAL:.0f} días")

##### 4. Stock y tránsito

- **Stock hoy:** `stock.quant` en ubicaciones internas de TVL y TATSA (por llave).
- **Tránsito:** OC pendientes de entrega y RQ sin OC del Excel del saldo (unidades).

In [ ]:
stock = []
for n, e in EMPRESAS.items():
    stock += [{"Almacen": n, "Llave": PRODUCTO.get(m2o_id(q["product_id"]), {}).get("Llave"), "Cantidad": q["quantity"]}
              for q in leer_todo("stock.quant", [["company_id", "=", e], ["location_id.usage", "=", "internal"], ["quantity", ">", 0]],
                                 ["product_id", "quantity"], [e])]
STOCK = pd.DataFrame(stock).pivot_table(index="Llave", columns="Almacen", values="Cantidad", aggfunc="sum", fill_value=0.0)
STOCK = STOCK.reindex(columns=list(EMPRESAS), fill_value=0.0).add_prefix("Stock_")
STOCK["Stock_Hoy"] = STOCK.sum(axis=1)

oc_pend = pd.read_excel(ARCHIVO_SALDO, sheet_name="OC_PENDIENTE_ENTREGA", engine="calamine")
rq_soc = pd.read_excel(ARCHIVO_SALDO, sheet_name="RQ_SIN_OC", engine="calamine")
TRANSITO = pd.DataFrame({
    "Transito_OC": oc_pend.assign(Llave=oc_pend["Cod_Producto"].map(LLAVE_DE_CODIGO)).groupby("Llave")["Cant_Pendiente"].sum(),
    "Transito_RQ": rq_soc.assign(Llave=rq_soc["Cod_Producto"].map(LLAVE_DE_CODIGO)).groupby("Llave")["Cant_Pendiente"].sum()}).fillna(0.0)
print(f"Llaves con stock: {len(STOCK):,} | con tránsito: {len(TRANSITO):,}")

##### 5. Clasificación ABC‑XYZ, lead time y punto de reorden

La misma función se aplica a cada **tamaño de bus** (12, 18 y 9 m), a **toda la flota** (los tres juntos) y al **consumo interno de TVL**:
- Serie mensual (unidades y valor) de los 12 meses cerrados, con meses sin consumo = 0.
- ABC por valor dentro del conjunto analizado; XYZ por CV de las unidades mensuales.
- Se agregan lead time, stock de seguridad, punto de reorden, stock, tránsito y compras.
- **Por tamaño:** ABC‑XYZ dentro de ese tamaño; el stock de seguridad y el punto de reorden se calculan con la demanda de ese tamaño (lo que necesita esa flota). El stock, el tránsito y las compras son de la llave y se comparten entre tamaños, por eso el estado (bajo reorden, sobrestock) se mide en **toda la flota**.

In [ ]:
def serie_mensual(df, claves):
    """Unidades y valor por claves + mes en la ventana (meses sin consumo = 0)."""
    # Todas las claves con consumo desde HISTORIA_DESDE (las que no consumieron en la ventana quedan en 0: sin rotación)
    todas = df.groupby(claves).size().index
    d = df[df["Periodo_Mes"].isin(VENTANA)].groupby(claves + ["Periodo_Mes"])[["Cantidad", "Valor"]].sum()
    und = d["Cantidad"].unstack("Periodo_Mes").reindex(index=todas, columns=VENTANA, fill_value=0.0).fillna(0.0).clip(lower=0)
    val = d["Valor"].unstack("Periodo_Mes").reindex(index=todas, columns=VENTANA, fill_value=0.0).fillna(0.0).clip(lower=0)
    return und, val


def clasificar(df, claves, grupo_abc=None):
    """ABC (por valor 12 meses, dentro de cada grupo_abc) y XYZ (CV mensual) por claves."""
    und, val = serie_mensual(df, claves)
    c = pd.DataFrame(index=und.index)
    c["Consumo_Und_12M"] = und.sum(axis=1)
    c["Consumo_Valor_12M"] = val.sum(axis=1)
    c["Prom_Mensual"] = und.mean(axis=1)
    c["Prom_6M"] = und[VENTANA[-6:]].mean(axis=1)
    c["Desv_Mensual"] = und.std(axis=1, ddof=0)
    c["CV"] = (c["Desv_Mensual"] / c["Prom_Mensual"]).replace([np.inf, -np.inf], np.nan)
    c["Meses_Con_Consumo"] = (und > 0).sum(axis=1)
    c["Ultimo_Consumo"] = df[df["Cantidad"] > 0].groupby(claves)["Fecha"].max().reindex(und.index).dt.strftime("%Y-%m-%d")
    c = c.reset_index()
    grupos = [grupo_abc] if grupo_abc else []
    c = c.sort_values(grupos + ["Consumo_Valor_12M"], ascending=[True] * len(grupos) + [False])
    total = c.groupby(grupos)["Consumo_Valor_12M"].transform("sum") if grupos else c["Consumo_Valor_12M"].sum()
    acum = (c.groupby(grupos)["Consumo_Valor_12M"].cumsum() if grupos else c["Consumo_Valor_12M"].cumsum()) / total
    c["Particip_Acum"] = acum
    c["ABC"] = np.select([c["Consumo_Valor_12M"] <= 0, acum <= ABC_CORTES[0], acum <= ABC_CORTES[1]], ["-", "A", "B"], default="C")
    c["XYZ"] = np.select([c["Consumo_Valor_12M"] <= 0, c["CV"] < XYZ_CORTES[0], c["CV"] <= XYZ_CORTES[1]], ["-", "X", "Y"], default="Z")
    c["ABC_XYZ"] = np.where(c["ABC"] == "-", "SIN CONSUMO 12M", c["ABC"] + "-" + c["XYZ"])
    return c


def tramo_lead_time(dias):
    return next(t for limite, t in TRAMOS_LEAD_TIME if dias <= limite)


def abastecimiento(c):
    """Agrega datos de la llave, compras, stock, tránsito, lead time, stock de seguridad y punto de reorden."""
    c = c.join(INFO_LLAVE, on="Llave").join(COMPRAS_LLAVE, on="Llave").join(STOCK, on="Llave").join(TRANSITO, on="Llave")
    c["Codigos"] = c["Llave"].map(CODIGOS_LLAVE)
    for col in ["Stock_Hoy", "Transito_OC", "Transito_RQ"] + [f"Stock_{n}" for n in EMPRESAS]:
        c[col] = c[col].fillna(0.0)
    c["Importacion"] = c["Importacion"].fillna("Sin compras")
    c["Lead_Time_Dias"] = c["Lead_Time_Mediana"].fillna(LT_GENERAL)
    c["Origen_Lead_Time"] = np.where(c["Lead_Time_Mediana"].notna(), "Propio", "General (sin recepciones)")
    c["Tramo_Lead_Time"] = c["Lead_Time_Dias"].map(tramo_lead_time)
    z = c["ABC"].map(Z_SERVICIO).fillna(0.0)
    c["Stock_Seguridad"] = z * c["Desv_Mensual"] * np.sqrt(c["Lead_Time_Dias"] / 30)
    c["Punto_Reorden"] = c["Prom_Mensual"] * c["Lead_Time_Dias"] / 30 + c["Stock_Seguridad"]
    c["Posicion"] = c["Stock_Hoy"] + c["Transito_OC"] + c["Transito_RQ"]
    c["Cobertura_Meses"] = np.where(c["Prom_Mensual"] > 0, c["Stock_Hoy"] / c["Prom_Mensual"].where(c["Prom_Mensual"] > 0, 1),
                                    np.where(c["Stock_Hoy"] > 0, np.inf, 0.0))
    c["Sugerido_Reponer"] = np.ceil((c["Punto_Reorden"] + c["Prom_Mensual"] - c["Posicion"]).clip(lower=0) - 1e-9).clip(lower=0)
    c["Estado"] = np.select(
        [(c["Consumo_Und_12M"] <= 0) & (c["Stock_Hoy"] > 0), c["Consumo_Und_12M"] <= 0,
         (c["Posicion"] < c["Punto_Reorden"]) & (c["Prom_Mensual"] > 0) & (c["ABC_XYZ"] == "C-Z"),
         (c["Posicion"] < c["Punto_Reorden"]) & (c["Prom_Mensual"] > 0), c["Cobertura_Meses"] > MESES_SOBRESTOCK],
        ["SIN ROTACIÓN", "SIN CONSUMO 12M", "C-Z: COMPRA A PEDIDO", "BAJO PUNTO DE REORDEN", "SOBRESTOCK"], default="OK")
    # Stock valorizado al costo medio del consumo (sin IGV); si no hubo consumo, al último precio de compra sin IGV
    costo = (c["Consumo_Valor_12M"] / c["Consumo_Und_12M"].where(c["Consumo_Und_12M"] > 0)).fillna(c["Precio_Ultimo"] / 1.18)
    c["Valor_Stock"] = c["Stock_Hoy"] * costo.fillna(0.0)
    return c


"""
    BUSES: UN ANÁLISIS POR CADA TAMAÑO Y OTRO DE TODA LA FLOTA
"""
POR_TAMANO = {}
for f in FLOTAS:
    c = abastecimiento(clasificar(buses[buses["Tamaño_Bus"] == f], ["Llave"]))
    c["Tamaño_Bus"], c["Buses"] = f, BUSES[f]
    c["Und_Bus_Mes"] = c["Prom_Mensual"] / BUSES[f]
    POR_TAMANO[f] = c
por_tamano = pd.concat(POR_TAMANO.values(), ignore_index=True)

consolidado = abastecimiento(clasificar(buses, ["Llave"]))
consolidado["Tamaño_Bus"], consolidado["Buses"] = "TODA LA FLOTA", BUSES["TODA LA FLOTA"]
consolidado["Und_Bus_Mes"] = consolidado["Prom_Mensual"] / BUSES["TODA LA FLOTA"]
k12 = buses[buses["Periodo_Mes"].isin(VENTANA)]
tipo = k12.groupby(["Llave", "Tipo_Mantto"])["Cantidad"].sum().unstack(fill_value=0.0)
consolidado["Pct_Preventivo"] = consolidado["Llave"].map((tipo.get("Preventivo", 0) / tipo.sum(axis=1)).replace([np.inf, -np.inf], np.nan))

"""
    CONSUMO INTERNO TVL
"""
tvl_interno = abastecimiento(clasificar(interno, ["Llave"]))

"""
    SERIE MENSUAL (DESDE 2024) PARA SEGUIMIENTO
"""
# Meses en filas: una fila por tamaño (y toda la flota) + mes + llave
por_mes = buses.groupby(["Tamaño_Bus", "Periodo_Mes", "Llave"])[["Cantidad", "Valor"]].sum().reset_index()
flota_mes = buses.groupby(["Periodo_Mes", "Llave"])[["Cantidad", "Valor"]].sum().reset_index().assign(Tamaño_Bus="TODA LA FLOTA")
mensual_buses = pd.concat([por_mes, flota_mes], ignore_index=True).rename(columns={"Cantidad": "Consumo_Und", "Valor": "Consumo_Valor"})
mensual_buses = mensual_buses[mensual_buses["Periodo_Mes"] <= ULTIMO_CERRADO]
mensual_buses["Mes"] = mensual_buses["Periodo_Mes"].astype(str)
mensual_buses["Producto_Grupo"] = mensual_buses["Llave"].map(INFO_LLAVE["Producto_Grupo"])
mensual_buses["Buses"] = mensual_buses["Tamaño_Bus"].map(BUSES)
mensual_buses["Und_Bus"] = mensual_buses["Consumo_Und"] / mensual_buses["Buses"]
orden_tam = {f: i for i, f in enumerate(FLOTAS + ["TODA LA FLOTA"])}
mensual_buses = mensual_buses.assign(_o=mensual_buses["Tamaño_Bus"].map(orden_tam)).sort_values(["_o", "Mes", "Consumo_Valor"], ascending=[True, True, False])
mensual_buses = mensual_buses[["Tamaño_Bus", "Mes", "Llave", "Producto_Grupo", "Consumo_Und", "Consumo_Valor", "Buses", "Und_Bus"]].reset_index(drop=True)
# Meses en filas: una fila por mes + llave
mensual_interno = interno[interno["Periodo_Mes"] <= ULTIMO_CERRADO].groupby(["Periodo_Mes", "Llave"])[["Cantidad", "Valor"]].sum().reset_index()
mensual_interno = mensual_interno.rename(columns={"Cantidad": "Consumo_Und", "Valor": "Consumo_Valor"})
mensual_interno["Mes"] = mensual_interno["Periodo_Mes"].astype(str)
mensual_interno["Producto_Grupo"] = mensual_interno["Llave"].map(INFO_LLAVE["Producto_Grupo"])
mensual_interno = mensual_interno.sort_values(["Mes", "Consumo_Valor"], ascending=[True, False])
mensual_interno = mensual_interno[["Mes", "Llave", "Producto_Grupo", "Consumo_Und", "Consumo_Valor"]].reset_index(drop=True)

pd.set_option("display.float_format", "{:,.0f}".format)
print(consolidado.groupby(["ABC", "XYZ"]).agg(Llaves=("Llave", "size"), Valor_12M=("Consumo_Valor_12M", "sum")).to_string())
print(consolidado.groupby("Estado").agg(Llaves=("Llave", "size"), Valor_Stock=("Valor_Stock", "sum")).to_string())

##### 6. Resumen

- **Matriz ABC‑XYZ:** llaves y valor consumido en 12 meses, para el acumulado de buses, cada tamaño y el consumo interno TVL.
- **Por tramo de lead time e importación:** llaves, valor consumido, stock y cuántas están bajo el punto de reorden.

In [ ]:
def matriz(c, nombre):
    m = c[c["ABC"] != "-"].pivot_table(index="ABC", columns="XYZ", values="Consumo_Valor_12M", aggfunc=["size", "sum"], fill_value=0)
    filas = []
    for abc in ["A", "B", "C"]:
        for xyz in ["X", "Y", "Z"]:
            filas.append({"Conjunto": nombre, "ABC": abc, "XYZ": xyz,
                          "Llaves": int(m[("size", xyz)].get(abc, 0)) if ("size", xyz) in m else 0,
                          "Valor_12M": float(m[("sum", xyz)].get(abc, 0.0)) if ("sum", xyz) in m else 0.0})
    df = pd.DataFrame(filas)
    df["Particip_Valor"] = df["Valor_12M"] / df["Valor_12M"].sum() if df["Valor_12M"].sum() else 0.0
    return df


resumen_abc = pd.concat([matriz(consolidado, "Buses acumulado")]
                        + [matriz(por_tamano[por_tamano["Tamaño_Bus"] == f], f"Buses {f.lower()}") for f in FLOTAS]
                        + [matriz(tvl_interno, "TVL consumo interno")], ignore_index=True)


def resumen_tramos(c, nombre):
    r = c[c["Consumo_Und_12M"] > 0].groupby(["Tramo_Lead_Time", "Importacion"]).agg(
        Llaves=("Llave", "size"), Valor_12M=("Consumo_Valor_12M", "sum"), Valor_Stock=("Valor_Stock", "sum"),
        Bajo_Reorden=("Estado", lambda s: int((s == "BAJO PUNTO DE REORDEN").sum())),
        Lead_Time_Prom=("Lead_Time_Dias", "mean")).reset_index()
    orden = {t: i for i, (_, t) in enumerate(TRAMOS_LEAD_TIME)}
    return r.assign(Conjunto=nombre, _o=r["Tramo_Lead_Time"].map(orden)).sort_values(["_o", "Importacion"]).drop(columns="_o")


resumen_lt = pd.concat([resumen_tramos(consolidado, "Buses acumulado"), resumen_tramos(tvl_interno, "TVL consumo interno")], ignore_index=True)
resumen_lt = resumen_lt[["Conjunto", "Tramo_Lead_Time", "Importacion", "Llaves", "Valor_12M", "Valor_Stock", "Bajo_Reorden", "Lead_Time_Prom"]]
resumen_tam = pd.concat([por_tamano, consolidado]).groupby("Tamaño_Bus", sort=False).agg(
    Llaves=("Consumo_Und_12M", lambda s: int((s > 0).sum())), Valor_12M=("Consumo_Valor_12M", "sum"), Und_12M=("Consumo_Und_12M", "sum")).reset_index()
resumen_tam["Buses"] = resumen_tam["Tamaño_Bus"].map(BUSES)
resumen_tam["Valor_Bus_Mes"] = resumen_tam["Valor_12M"] / resumen_tam["Buses"] / MESES_ANALISIS
print(resumen_tam.to_string())
print(resumen_lt.to_string())

##### 7. Exportar a Excel

`Analisis_Repuestos.xlsx` (con `EXPORTAR_EXCEL = True`): RESUMEN_ABC_XYZ, RESUMEN_LEAD_TIME, RESUMEN_TAMAÑO, BUSES_FLOTA_COMPLETA, BUSES_12M, BUSES_18M, BUSES_9M, MENSUAL_BUSES (meses en filas), TVL_CONSUMO_INTERNO y MENSUAL_TVL_INTERNO (meses en filas).
Diseño: sin cuadrícula, Consolas 8, soles con negativos en rojo, encabezados por color (azul = identificación, ámbar = consumo, morado = clasificación, azul acero = abastecimiento e inventario, rojo = estado, verde = compras, gris = notas), filtro y paneles congelados.

In [ ]:
FUENTE, FUENTE_T = Font(name="Consolas", size=8), Font(name="Consolas", size=8, bold=True, color="FFFFFF")
F_SOLES, F_CANT, F_PCT, F_DIAS = '"S/" #,##0.00;[Red]-"S/" #,##0.00', "#,##0.00;[Red]-#,##0.00", "0.0%", "#,##0"
COLOR = {"id": "1F4E78", "consumo": "BF8F00", "clase": "7030A0", "inv": "2E75B6", "estado": "C00000", "compra": "375623", "nota": "595959"}
TEXTO = {"Producto": 34, "Producto_Grupo": 34, "Categoria": 30, "Codigos": 30, "Ultimo_Proveedor": 28, "Proveedor_Principal": 28,
         "Proveedores_24M": 50, "Familia": 18}
CENTRADAS = {"Mes", "Llave", "ABC", "XYZ", "ABC_XYZ", "Estado", "Importacion", "Tamaño_Bus", "Tramo_Lead_Time", "UdM", "Moneda_Ultima",
             "Ultima_OC", "Fecha_Ultima_Compra", "Ultimo_Consumo", "Conjunto", "Origen_Lead_Time", "Tipo_Producto"}


def grupo_col(c):
    if c.startswith(("Consumo", "Prom", "Desv", "Meses_", "Und_", "Ultimo_Consumo", "Pct_", "Valor_Bus")) or c[:4].isdigit():
        return "consumo"
    if c.startswith(("ABC", "XYZ", "CV", "Particip")):
        return "clase"
    if c.startswith(("Stock", "Transito", "Posicion", "Punto", "Lead", "Tramo", "Cobertura", "Origen_Lead", "Sugerido", "Valor_Stock", "Recepciones")):
        return "inv"
    if c.startswith(("Estado", "Bajo")):
        return "estado"
    if c.startswith(("Importacion", "Fecha_Ultima", "Ultima_OC", "Ultimo_Proveedor", "Moneda", "Precio", "Compras", "OC_", "Proveedor", "N_Proveedores")):
        return "compra"
    return "id"


def formato_col(c):
    if c.startswith(("Particip", "Pct_")):
        return F_PCT
    if c.startswith(("Lead_Time", "Meses_", "Llaves", "Buses", "OC_", "N_Proveedores", "Recepciones", "Bajo_")):
        return F_DIAS
    if c.startswith(("Consumo_Valor", "Valor", "Precio", "Compras_Valor")):
        return F_SOLES
    return F_CANT


def ancho(df, c, fmt):
    if c in TEXTO:
        return TEXTO[c]
    v = df[c].replace([np.inf, -np.inf], np.nan).dropna()
    if fmt == F_SOLES:
        largo = max([len(f"S/ {x:,.2f}") + 1 for x in v], default=0)
    elif fmt in (F_CANT, F_DIAS):
        largo = max([len(f"{x:,.2f}") for x in v], default=0)
    elif fmt == F_PCT:
        largo = 7
    else:
        largo = max([len(str(x)) for x in v], default=0)
    return min(max(largo, len(c) * 0.85, 6) + 2, 28)


def dar_formato(hoja, df):
    hoja.sheet_view.showGridLines = False
    fmt, ali = {}, {}
    for i, c in enumerate(df.columns, start=1):
        celda = hoja.cell(1, i)
        celda.font, celda.border = FUENTE_T, Border()
        celda.fill = PatternFill("solid", fgColor=COLOR[grupo_col(c)])
        celda.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        num = pd.api.types.is_numeric_dtype(df[c]) and not pd.api.types.is_bool_dtype(df[c])
        if num:
            fmt[i] = formato_col(c)
        ali[i] = Alignment(horizontal="right" if num else "center" if c in CENTRADAS else "left", vertical="center")
        hoja.column_dimensions[get_column_letter(i)].width = ancho(df, c, fmt.get(i))
    for fila in hoja.iter_rows(min_row=2):
        for celda in fila:
            celda.font = FUENTE
            celda.alignment = ali[celda.column]
            if celda.column in fmt:
                celda.number_format = fmt[celda.column]
    hoja.row_dimensions[1].height = 30
    hoja.freeze_panes = "C2" if "Producto" in df.columns[:3] or "Producto_Grupo" in df.columns[:3] else "A2"
    if len(df):
        hoja.auto_filter.ref = hoja.dimensions


COLS_ABAST = (["Codigos", "Producto_Grupo", "Familia", "Categoria", "UdM", "Tipo_Producto"]
              + ["Consumo_Und_12M", "Consumo_Valor_12M", "Prom_Mensual", "Prom_6M", "Desv_Mensual", "CV", "Meses_Con_Consumo", "Ultimo_Consumo",
                 "Particip_Acum", "ABC", "XYZ", "ABC_XYZ"]
              + ["Lead_Time_Dias", "Lead_Time_P90", "Origen_Lead_Time", "Recepciones_Medidas", "Tramo_Lead_Time", "Stock_Seguridad", "Punto_Reorden"]
              + [f"Stock_{n}" for n in EMPRESAS] + ["Stock_Hoy", "Transito_OC", "Transito_RQ", "Posicion", "Cobertura_Meses", "Valor_Stock", "Sugerido_Reponer", "Estado"]
              + ["Importacion", "Fecha_Ultima_Compra", "Ultima_OC", "Ultimo_Proveedor", "Moneda_Ultima", "Precio_Ultimo", "Precio_Prom_12M",
                 "Compras_Und_12M", "Compras_Valor_12M", "OC_12M", "Proveedor_Principal", "Participacion_Principal", "N_Proveedores_24M", "Proveedores_24M"])
# Toda la flota: se trata y clasifica como una sola flota (sin columnas por tamaño)
cols_bus = ["Llave"] + COLS_ABAST[:6] + ["Pct_Preventivo"] + COLS_ABAST[6:]
# Por tamaño: el estado y la posición se miden en toda la flota (stock compartido); aquí va el requerimiento de esa flota
QUITAR_TAMANO = {"Posicion", "Cobertura_Meses", "Sugerido_Reponer", "Estado", "Valor_Stock"}
cols_tam = ["Llave"] + COLS_ABAST[:6] + ["Buses", "Und_Bus_Mes"] + [c for c in COLS_ABAST[6:] if c not in QUITAR_TAMANO]
cols_bus = cols_bus[:7] + ["Buses", "Und_Bus_Mes"] + cols_bus[7:]
orden_estado = {"BAJO PUNTO DE REORDEN": 0, "OK": 1, "SOBRESTOCK": 2, "SIN ROTACIÓN": 3, "SIN CONSUMO": 4}

if EXPORTAR_EXCEL:
    hojas = {
        "RESUMEN_ABC_XYZ": resumen_abc,
        "RESUMEN_LEAD_TIME": resumen_lt,
        "RESUMEN_TAMAÑO": resumen_tam,
        "BUSES_FLOTA_COMPLETA": consolidado[cols_bus].sort_values("Consumo_Valor_12M", ascending=False),
        **{f"BUSES_{f.split()[0]}M": POR_TAMANO[f][cols_tam].sort_values("Consumo_Valor_12M", ascending=False) for f in FLOTAS},
        "MENSUAL_BUSES": mensual_buses,
        "TVL_CONSUMO_INTERNO": tvl_interno[["Llave"] + COLS_ABAST].sort_values("Consumo_Valor_12M", ascending=False),
        "MENSUAL_TVL_INTERNO": mensual_interno,
    }
    with pd.ExcelWriter(ARCHIVO_EXCEL, engine="openpyxl") as w:
        for nombre, df in hojas.items():
            df = df.replace([np.inf, -np.inf], np.nan)
            df.to_excel(w, sheet_name=nombre, index=False)
            dar_formato(w.sheets[nombre], df)
    print(f"Excel generado: {ARCHIVO_EXCEL}", {h: len(d) for h, d in hojas.items()})
else:
    print("Exportación desactivada (EXPORTAR_EXCEL = False)")